# Segmentasi Pelanggan LRFM — Penjualan April-September 2020

Notebook ini dibangun **di atas data bersih** (`bersih/penjualan_bersih.csv`), hasil
`02_pembersihan_dan_anomali.ipynb` — bukan file mentah. Ini bukan pilihan sembarangan:
LRFM menghitung agregat per pelanggan, dan agregat itu rapuh terhadap masalah yang
justru sudah dibereskan di notebook sebelumnya:

- **Duplikat** yang tidak ditandai akan menggelembungkan *Frequency* satu pelanggan.
- **Harga hilang** yang tidak dipulihkan (K03a) akan mengecilkan *Monetary* pelanggan
  yang justru membeli dalam jumlah besar.
- **Identitas pelanggan pecah** (`ADEK FROZEN` vs `C - ADEK FROZEN`) akan membuat satu
  pelanggan terhitung sebagai dua entitas terpisah, masing-masing dengan LRFM yang salah.

**Apa itu LRFM**

Perluasan dari RFM klasik dengan satu dimensi tambahan:

| Huruf | Pertanyaan | Dihitung dari |
|---|---|---|
| **L** — Length | Sudah berapa lama pelanggan ini dikenal? | rentang hari pembelian pertama sampai terakhir |
| **R** — Recency | Berapa lama sejak transaksi terakhirnya? | hari sejak `TGL` terakhir sampai tanggal analisis |
| **F** — Frequency | Seberapa sering dia bertransaksi? | jumlah hari berbeda dia tercatat membeli |
| **M** — Monetary | Berapa besar nilai belanjanya? | total `OMZET` sepanjang periode |

`L` yang membedakan LRFM dari RFM biasa: dua pelanggan bisa punya R, F, M yang mirip,
tapi salah satunya pelanggan lama yang mulai jarang beli, sedangkan satunya pelanggan
baru yang belum sempat menunjukkan pola. Tanpa `L`, keduanya akan disegmentasi sama.

**Alur**

| Bagian | Isi |
|---|---|
| 1 | Muat data bersih & tetapkan cakupan (unit pelanggan mana yang layak dianalisis) |
| 2 | Hitung L, R, F, M mentah per pelanggan |
| 3 | Skoring 1-5 per dimensi (kuantil, bukan ambang tetap) |
| 4 | Peta segmen — aturan eksplisit berbasis skor R x F |
| 5 | Profil & visualisasi tiap segmen |
| 6 | Ekspor


## 1. Muat data bersih & tetapkan cakupan

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

FILE_BERSIH = os.path.join('bersih', 'penjualan_bersih.csv')
DIR_KELUARAN = 'bersih'

WARNA_SEGMEN = {
    'Juara': '#2E7D32', 'Pelanggan Setia': '#66BB6A', 'Potensi Setia': '#9CCC65',
    'Pelanggan Baru': '#4C72B0', 'Menjanjikan': '#64B5CD',
    'Butuh Perhatian': '#DD8452', 'Akan Tertidur': '#E8A87C',
    'Berisiko': '#C44E52', 'Jangan Sampai Hilang': '#8C4351', 'Tertidur': '#9AA5B1',
}

d = pd.read_csv(FILE_BERSIH, parse_dates=['TGL', 'BULAN'])
print('Dimensi data bersih:', d.shape)
print(f'Periode: {d["TGL"].min():%d %b %Y} s/d {d["TGL"].max():%d %b %Y}')
print(f'Pelanggan unik (mentah): {d["PELANGGAN"].nunique()}')
d.head()

`CASH` bukan satu pelanggan — ia label transaksi tunai/walk-in yang dipakai untuk
banyak pembeli berbeda yang tidak tercatat identitasnya. Menyertakannya ke LRFM berarti
menjumlahkan riwayat ratusan orang tak terkait jadi satu "pelanggan super", yang akan
selalu muncul sebagai *Juara* semu dan merusak skala kuantil untuk pelanggan lain.
Ia dikeluarkan dari segmentasi, tetapi nilainya tetap dilaporkan terpisah supaya tidak
hilang dari total omzet.

In [ ]:
PSEUDO_PELANGGAN = ['CASH']   # bukan identitas pelanggan asli -> dikeluarkan dari LRFM

tunai = d[d['PELANGGAN'].isin(PSEUDO_PELANGGAN)]
valid = d[~d['PELANGGAN'].isin(PSEUDO_PELANGGAN)].copy()

print(f'Baris tunai/walk-in dikeluarkan : {len(tunai):,} '
      f'(Rp {tunai["OMZET"].sum():,.0f}, {tunai["OMZET"].sum() / d["OMZET"].sum() * 100:.1f}% omzet)')
print(f'Baris & pelanggan untuk LRFM    : {len(valid):,} baris, '
      f'{valid["PELANGGAN"].nunique()} pelanggan')

n_sekali = int((valid.groupby('PELANGGAN')['TGL'].nunique() == 1).sum())
print(f'Pelanggan dengan hanya 1 hari transaksi tercatat: {n_sekali} '
      f'— L mereka otomatis 0. Itu bukan kesalahan, hanya belum ada riwayat kedua.')

## 2. Hitung L, R, F, M mentah per pelanggan

Tanggal analisis diambil sehari setelah transaksi terakhir di seluruh dataset
(`2020-09-30 + 1`), bukan tanggal hari ini — supaya Recency mengukur posisi pelanggan
**relatif terhadap akhir periode data**, bukan relatif terhadap kapan notebook ini
kebetulan dijalankan.

`F` dihitung dari **jumlah hari berbeda** pelanggan itu bertransaksi, bukan jumlah baris.
Satu nota dengan 5 SKU sekaligus bukan berarti pelanggan itu "5 kali lebih sering"
belanja — itu tetap satu kunjungan. Menghitung baris akan menghukum pelanggan yang
beli banyak jenis barang sekaligus, padahal itu justru perilaku baik.

In [ ]:
TGL_ANALISIS = d['TGL'].max() + pd.Timedelta(days=1)
print(f'Tanggal analisis (H+1 dari transaksi terakhir): {TGL_ANALISIS:%d %b %Y}')

lrfm = valid.groupby('PELANGGAN').agg(
    TANGGAL_PERTAMA=('TGL', 'min'),
    TANGGAL_TERAKHIR=('TGL', 'max'),
    F=('TGL', 'nunique'),
    N_BARIS=('TGL', 'size'),
    M=('OMZET', 'sum'),
    M_RATA_PER_HARI=('OMZET', lambda s: s.sum() / s.index.size),
)
lrfm['L'] = (lrfm['TANGGAL_TERAKHIR'] - lrfm['TANGGAL_PERTAMA']).dt.days
lrfm['R'] = (TGL_ANALISIS - lrfm['TANGGAL_TERAKHIR']).dt.days
lrfm = lrfm.reset_index()

print(f'\n{len(lrfm)} pelanggan diberi nilai LRFM.')
display(lrfm[['L', 'R', 'F', 'M']].describe().round(1))

In [ ]:
# Sebaran tiap dimensi condong ke kanan (segelintir pelanggan sangat besar), khas
# data transaksi -- alasan yang sama dengan kenapa notebook 02 memakai median+MAD,
# bukan rata-rata, untuk pencilan harga.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, kol, judul in zip(axes.flat, ['L', 'R', 'F', 'M'],
                          ['Length (hari)', 'Recency (hari)', 'Frequency (hari beli)',
                           'Monetary (Rp)']):
    ax.hist(lrfm[kol], bins=30, color='#4C72B0', alpha=0.85)
    ax.axvline(lrfm[kol].median(), color=WARNA_SEGMEN['Berisiko'], ls='--', lw=1.5,
               label=f'median = {lrfm[kol].median():,.0f}')
    ax.set_title(judul)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Skoring 1-5 per dimensi

Skor dibagi lewat **kuantil** (tiap dimensi dipecah jadi 5 kelompok berisi jumlah
pelanggan yang kurang lebih sama), bukan ambang tetap seperti "F >= 10 = sering". Data
6 bulan ini sudah menunjukkan sebaran yang timpang (lihat histogram di atas); ambang
tetap yang masuk akal untuk bulan pertama belum tentu masuk akal begitu periode datanya
berubah. Kuantil menyesuaikan diri terhadap sebaran data itu sendiri.

Arah skor `R` dibalik: Recency yang **kecil** (baru saja beli) mendapat skor **5**,
karena secara bisnis "baru saja aktif" adalah kondisi baik — kebalikan dari L, F, M
yang searah (angka besar = skor besar).

In [ ]:
def skor_kuantil(seri, arah_naik=True, k=5):
    '''Bagi seri ke k kelompok berisi jumlah anggota sama lewat kuantil.

    rank(method='first') dipakai sebelum qcut supaya nilai yang sama (mis. banyak
    pelanggan dengan F=1) tetap bisa dipecah ke kelompok berbeda tanpa error
    "duplicate bin edges" -- urutannya tetap stabil karena rank memutus seri
    berdasarkan urutan kemunculan, bukan mengarang nilai.
    '''
    peringkat = seri.rank(method='first')
    label = range(1, k + 1) if arah_naik else range(k, 0, -1)
    return pd.qcut(peringkat, k, labels=list(label)).astype(int)


lrfm['SKOR_L'] = skor_kuantil(lrfm['L'], arah_naik=True)
lrfm['SKOR_R'] = skor_kuantil(lrfm['R'], arah_naik=False)
lrfm['SKOR_F'] = skor_kuantil(lrfm['F'], arah_naik=True)
lrfm['SKOR_M'] = skor_kuantil(lrfm['M'], arah_naik=True)

display(lrfm.groupby('SKOR_R')[['R']].agg(['min', 'max', 'size']))

## 4. Peta segmen

Segmentasi dibuat lewat **satu tabel aturan eksplisit** berbasis kombinasi `SKOR_R` dan
`SKOR_F` — pola yang sama dengan kebijakan pembersihan di notebook 02: keputusan
ditulis di satu tempat, bisa dibaca ulang, dan bisa diubah tanpa menyentuh kode di
sekitarnya. `M` dan `L` sengaja **tidak** ikut menentukan nama segmen; keduanya dipakai
sebagai lensa tambahan saat membaca profil tiap segmen di Bagian 5, supaya segmennya
tetap gampang dijelaskan ("baru & sering beli" vs "baru & sering beli, TAPI nilainya
kecil") tanpa membuat aturannya sendiri jadi rumit.

R dan F dipilih sebagai dasar karena keduanya menjawab pertanyaan paling mendesak untuk
tindakan: *masih aktif?* dan *seberapa lekat kebiasaannya?* — dua hal yang paling cepat
berubah dan paling relevan untuk keputusan retensi.

In [ ]:
# Peta standar RF (mis. dari kerangka segmentasi RFM populer), disesuaikan penamaannya.
# Kunci = pola regex atas string "SKOR_R + SKOR_F" (mis. '54' = R skor 5, F skor 4).
PETA_SEGMEN = {
    r'[1-2][1-2]': 'Tertidur',
    r'[1-2][3-4]': 'Berisiko',
    r'[1-2]5':      'Jangan Sampai Hilang',
    r'3[1-2]':      'Akan Tertidur',
    r'33':          'Butuh Perhatian',
    r'[3-4][4-5]':  'Pelanggan Setia',
    r'41':          'Menjanjikan',
    r'51':          'Pelanggan Baru',
    r'[4-5][2-3]':  'Potensi Setia',
    r'5[4-5]':      'Juara',
}

kode_rf = lrfm['SKOR_R'].astype(str) + lrfm['SKOR_F'].astype(str)
lrfm['SEGMEN'] = kode_rf.replace(PETA_SEGMEN, regex=True)

tak_terpetakan = ~lrfm['SEGMEN'].isin(PETA_SEGMEN.values())
assert not tak_terpetakan.any(), f'{tak_terpetakan.sum()} baris tidak kena aturan mana pun.'

urutan_segmen = ['Juara', 'Pelanggan Setia', 'Potensi Setia', 'Pelanggan Baru',
                 'Menjanjikan', 'Butuh Perhatian', 'Akan Tertidur', 'Berisiko',
                 'Jangan Sampai Hilang', 'Tertidur']
lrfm['SEGMEN'] = pd.Categorical(lrfm['SEGMEN'], categories=urutan_segmen, ordered=True)

print('Sebaran pelanggan per segmen:')
display(lrfm['SEGMEN'].value_counts().reindex(urutan_segmen).rename('pelanggan').to_frame())

## 5. Profil & visualisasi tiap segmen

In [ ]:
profil = (lrfm.groupby('SEGMEN', observed=True)
              .agg(pelanggan=('PELANGGAN', 'size'),
                   omzet_total=('M', 'sum'),
                   omzet_rata2=('M', 'mean'),
                   L_median=('L', 'median'),
                   R_median=('R', 'median'),
                   F_median=('F', 'median'))
              .reindex(urutan_segmen))
profil['pelanggan_%'] = (profil['pelanggan'] / profil['pelanggan'].sum() * 100).round(1)
profil['omzet_%'] = (profil['omzet_total'] / profil['omzet_total'].sum() * 100).round(1)
display(profil)

print(f'\nDicek: {profil["pelanggan_%"].sum():.0f}% pelanggan, '
      f'{profil["omzet_%"].sum():.0f}% omzet -- konsisten dengan seluruh pelanggan valid.')

In [ ]:
# 5.1 Jumlah pelanggan vs kontribusi omzet per segmen -- keduanya sengaja disandingkan
# karena bisa sangat berbeda arah: segmen kecil boleh jadi penyumbang omzet terbesar.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
tampil = profil.dropna(subset=['pelanggan']).loc[lambda x: x['pelanggan'] > 0]
warna = [WARNA_SEGMEN[s] for s in tampil.index]

axes[0].barh(tampil.index[::-1], tampil['pelanggan'][::-1], color=warna[::-1])
axes[0].set_title('Jumlah pelanggan per segmen')
axes[0].set_xlabel('Pelanggan')

axes[1].barh(tampil.index[::-1], tampil['omzet_total'][::-1] / 1e6, color=warna[::-1])
axes[1].set_title('Kontribusi omzet per segmen')
axes[1].set_xlabel('Omzet (juta Rp)')
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Peta R x F: setiap sel menunjukkan berapa pelanggan berada di kombinasi skor itu,
# dan warnanya mengikuti segmen yang menempel pada sel tersebut. Ini bentuk visual dari
# tabel PETA_SEGMEN di Bagian 4.
grid = lrfm.groupby(['SKOR_R', 'SKOR_F'], observed=True).size().unstack(fill_value=0)
grid = grid.reindex(index=range(5, 0, -1), columns=range(1, 6), fill_value=0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(grid.values, cmap='YlGnBu')
ax.set_xticks(range(5), range(1, 6))
ax.set_yticks(range(5), range(5, 0, -1))
ax.set_xlabel('Skor Frequency')
ax.set_ylabel('Skor Recency')
ax.set_title('Jumlah pelanggan per kombinasi skor R x F')
for i in range(5):
    for j in range(5):
        v = grid.values[i, j]
        ax.text(j, i, str(v), ha='center', va='center',
                color='white' if v > grid.values.max() * 0.6 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, label='pelanggan')
plt.tight_layout()
plt.show()

In [ ]:
# 5.3 Recency vs Frequency, tiap titik satu pelanggan, ukuran titik ~ Monetary,
# warna = segmen. Menunjukkan ketiga dimensi R/F/M sekaligus dalam satu pandangan.
fig, ax = plt.subplots(figsize=(11, 6.5))
for seg in urutan_segmen:
    sub = lrfm[lrfm['SEGMEN'] == seg]
    if sub.empty:
        continue
    ax.scatter(sub['R'], sub['F'], s=20 + (sub['M'] / lrfm['M'].max()) * 400,
               color=WARNA_SEGMEN[seg], alpha=0.75, edgecolor='white', lw=0.4, label=seg)
ax.set_xlabel('Recency (hari sejak transaksi terakhir)')
ax.set_ylabel('Frequency (hari beli berbeda)')
ax.set_title('Peta pelanggan: Recency x Frequency (ukuran titik ~ Monetary)')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8, title='Segmen')
plt.tight_layout()
plt.show()

In [ ]:
# 5.4 Sepuluh pelanggan bernilai tertinggi di tiap segmen prioritas -- daftar kerja
# untuk tim yang akan menindaklanjuti (retensi Juara, reaktivasi Berisiko, dst).
for seg in ['Juara', 'Berisiko', 'Jangan Sampai Hilang', 'Tertidur']:
    print(f'--- {seg} ---')
    display(lrfm[lrfm['SEGMEN'] == seg]
            .sort_values('M', ascending=False)
            [['PELANGGAN', 'L', 'R', 'F', 'M', 'TANGGAL_TERAKHIR']]
            .head(5))

## 6. Ekspor

In [ ]:
os.makedirs(DIR_KELUARAN, exist_ok=True)

KOLOM_EKSPOR = ['PELANGGAN', 'TANGGAL_PERTAMA', 'TANGGAL_TERAKHIR', 'L', 'R', 'F', 'M',
                'SKOR_L', 'SKOR_R', 'SKOR_F', 'SKOR_M', 'SEGMEN']
p_lrfm = os.path.join(DIR_KELUARAN, 'lrfm_pelanggan.csv')
lrfm[KOLOM_EKSPOR].to_csv(p_lrfm, index=False)

print(f'{p_lrfm:<35} {os.path.getsize(p_lrfm) / 1024:>8,.1f} KB')
print(f'{len(lrfm)} pelanggan diberi LRFM & segmen.')
print(f'Dikeluarkan dari segmentasi: {tunai["PELANGGAN"].nunique()} label tunai/walk-in '
      f'({", ".join(PSEUDO_PELANGGAN)}), senilai Rp {tunai["OMZET"].sum():,.0f}.')

garis = '-' * 70
print(f'\n{garis}')
print('RINGKASAN SEGMENTASI LRFM'.center(70))
print(garis)
top3 = profil.sort_values('omzet_total', ascending=False).head(3)
for seg, r in top3.iterrows():
    print(f'  {seg:<22}: {int(r["pelanggan"]):>3} pelanggan '
          f'({r["pelanggan_%"]:.0f}%)  ->  Rp {r["omzet_total"]:,.0f} omzet '
          f'({r["omzet_%"]:.0f}%)')
n_prioritas = int(profil.loc[['Berisiko', 'Jangan Sampai Hilang'], 'pelanggan'].sum())
omzet_prioritas = profil.loc[['Berisiko', 'Jangan Sampai Hilang'], 'omzet_total'].sum()
print(f'\n  Perlu tindakan retensi segera: {n_prioritas} pelanggan '
      f'(Berisiko + Jangan Sampai Hilang), Rp {omzet_prioritas:,.0f} omzet historis '
      f'yang berpotensi hilang.')
print(garis)